# Preprocess DegaFiles and Viz — Pancreas

A minimal end-to-end walkthrough of the Celldega pipeline on the public
**Xenium V1 Human Pancreas (FFPE)** dataset: download the raw data (only if it
isn't already present), run `dega.pre.main` to generate DegaFiles, then view the
result in a `Landscape` widget.

Data source: https://www.10xgenomics.com/datasets/ffpe-human-pancreas-with-xenium-multimodal-cell-segmentation-1-standard

In [ ]:
import celldega as dega

dega.__version__

## 1. Download raw Xenium data (only if needed)

Checks for the extracted data first, then the downloaded archive, and only
`curl`s / unzips what's missing — so re-running the notebook doesn't re-download
~9 GB every time.

In [ ]:
import subprocess
from pathlib import Path

sample = "Xenium_V1_human_Pancreas_FFPE_outs"
url = (
    "https://cf.10xgenomics.com/samples/xenium/2.0.0/"
    "Xenium_V1_human_Pancreas_FFPE/Xenium_V1_human_Pancreas_FFPE_outs.zip"
)

data_root_dir = Path("data/xenium_datasets")
data_dir = data_root_dir / sample
zip_path = data_root_dir / f"{sample}.zip"

# `experiment.xenium` is written at the top of every Xenium output bundle, so
# its presence is a reliable signal that the raw data is already extracted.
sentinel = data_dir / "experiment.xenium"

data_root_dir.mkdir(parents=True, exist_ok=True)

if sentinel.exists():
    print(f"Raw Xenium data already present at {data_dir} — skipping download.")
else:
    if zip_path.exists():
        print(f"Archive already downloaded at {zip_path} — skipping curl.")
    else:
        print(f"Downloading raw Xenium data -> {zip_path} ...")
        subprocess.run(["curl", "-L", url, "-o", str(zip_path)], check=True)

    print(f"Extracting {zip_path} -> {data_dir} ...")
    # -n: never overwrite existing files, so a partial re-run is safe.
    subprocess.run(["unzip", "-n", str(zip_path), "-d", str(data_dir)], check=True)
    print("Done.")

## 2. Run preprocessing → DegaFiles

`dega.pre.main` reads the raw Xenium output under `data_root_dir/sample` and
writes the DegaFiles landscape bundle to `path_dega_files`.

In [ ]:
tile_size = 250
image_tile_layer = "all"
path_dega_files = f"data/landscape_files/{sample}"

dega.pre.main(
    sample=sample,
    data_root_dir=str(data_root_dir),
    tile_size=tile_size,
    image_tile_layer=image_tile_layer,
    path_dega_files=path_dega_files,
    use_int_index=True,
)

## 3. Visualize the DegaFiles in a Landscape

By default this renders the `Landscape` widget from a public GitHub-hosted
copy of these same Pancreas DegaFiles, so the cell below works immediately —
without waiting on steps 1–2 to download/preprocess anything.

In [ ]:
base_url = (
    "https://raw.githubusercontent.com/broadinstitute/"
    "celldega_Xenium_human_Pancreas_FFPE/main/"
    "Landscape_Xenium_V1_human_Pancreas_FFPE_outs_webp"
)

landscape_ist = dega.viz.Landscape(
    technology="Xenium",
    base_url=base_url,
)

landscape_ist

### Using your own locally generated DegaFiles instead

To view the files you generated in step 2 (`path_dega_files`) rather than the
GitHub-hosted copy, serve them from a local server and point `base_url` there
instead of the URL above:

```python
server_address = dega.viz.get_local_server()
base_url = f"http://localhost:{server_address}/{path_dega_files}"

landscape_ist = dega.viz.Landscape(
    technology="Xenium",
    base_url=base_url,
)

landscape_ist
```